# 📐 Step 4 — Normalisation and Standardisation

**What this step does:**
- Takes the full feature matrix from Step 3 (log return + all indicators)
- Applies **MinMaxScaler** (scales to 0–1) and **StandardScaler** (zero mean, unit std)
- **Fits ONLY on training data** — test data is transformed using training statistics
- Saves both scalers so predictions can be inverse-transformed back to real prices

**Critical Rule — No Data Leakage:**
```
✅ CORRECT : fit_transform(train) → transform(test)
❌ WRONG   : fit_transform(all data) — this leaks future info into training
```

**Input file  :** `nifty50_step3_log_return.csv` (output from Step 3)

**Output files:**
- `nifty50_step4_minmax_scaled.csv` — MinMax scaled (0 to 1)
- `nifty50_step4_standard_scaled.csv` — Standard scaled (mean=0, std=1)
- `minmax_scaler.pkl` — saved MinMaxScaler for inverse transform later
- `standard_scaler.pkl` — saved StandardScaler for inverse transform later

---
### Instructions
1. Run **Cell 1** — install libraries
2. Run **Cell 2** — upload your Step 3 output CSV
3. Run **Cell 3** — define features and perform train/test split
4. Run **Cell 4** — apply MinMaxScaler (fit on train only)
5. Run **Cell 5** — apply StandardScaler (fit on train only)
6. Run **Cell 6** — verify no data leakage
7. Run **Cell 7** — full preview
8. Run **Cell 8** — save and download all output files

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install pandas numpy scikit-learn joblib -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Upload and Load Step 3 Output CSV
# ─────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from google.colab import files

print('Please upload your Step 3 output CSV file...')
print('(nifty50_step3_log_return.csv)')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f'\n✓ File uploaded: {filename}')

# Load CSV
df = pd.read_csv(filename)

# Parse dates
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

# Ensure all numeric columns are numeric
for col in df.columns:
    if col != 'Date':
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'\n✓ Rows loaded  : {len(df)} trading days')
print(f'✓ Date range   : {df["Date"].iloc[0].date()} → {df["Date"].iloc[-1].date()}')
print(f'✓ Columns      : {list(df.columns)}')
print(f'\nFirst 3 rows:')
print(df.head(3).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Define Feature Matrix and Train / Test Split
#
# Feature matrix includes:
#   LogReturn_Close — log return of Close price (from Step 3)
#   EMA_20          — 20-day EMA
#   EMA_50          — 50-day EMA
#   RSI_14          — 14-day RSI
#   BB_Upper        — Bollinger Upper Band
#   BB_Middle       — Bollinger Middle Band
#   BB_Lower        — Bollinger Lower Band
#   BB_Width        — Band width (volatility measure)
#   BB_Position     — Price position within bands
#   MACD_Line       — MACD Line
#   MACD_Signal     — Signal Line
#   MACD_Hist       — Histogram
#
# NOT included in feature matrix:
#   Date            — not a feature
#   Open, High, Low — not used in this model (Close-based model)
#   EMA_Cross       — binary signal, already 0/1, no scaling needed
#
# Train / Test Split: 80% train, 20% test — CHRONOLOGICAL (no shuffling)
# ─────────────────────────────────────────────────────────────────────────

# ── CONFIG ────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.80    # 80% train, 20% test

# ── Define feature columns ─────────────────────────────────────────────────
# Automatically detect available feature columns from the uploaded file
NON_FEATURE_COLS = ['Date', 'Open', 'High', 'Low', 'Close', 'EMA_Cross']
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]

print('=' * 60)
print('  FEATURE MATRIX DEFINITION')
print('=' * 60)
print(f'  Feature columns ({len(FEATURE_COLS)}):')
for col in FEATURE_COLS:
    print(f'    ✓ {col}')
print(f'\n  Excluded columns (not features):')
for col in NON_FEATURE_COLS:
    if col in df.columns:
        print(f'    — {col}')

# ── Drop rows with NaN in any feature column ────────────────────────────────
df_clean = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
dropped = len(df) - len(df_clean)
print(f'\n  Rows before NaN drop : {len(df)}')
print(f'  Rows dropped (NaN)   : {dropped}  (expected — first rows have no indicators)')
print(f'  Rows after NaN drop  : {len(df_clean)}')

# ── Chronological train/test split ─────────────────────────────────────────
split_idx  = int(len(df_clean) * TRAIN_RATIO)
train_df   = df_clean.iloc[:split_idx].copy()
test_df    = df_clean.iloc[split_idx:].copy()

print(f'\n  Train / Test Split (chronological — NO shuffling):')
print(f'    Train : {len(train_df)} rows  {train_df["Date"].iloc[0].date()} → {train_df["Date"].iloc[-1].date()}')
print(f'    Test  : {len(test_df)} rows  {test_df["Date"].iloc[0].date()} → {test_df["Date"].iloc[-1].date()}')
print(f'\n  ⚠️  Scaler will be fit ONLY on train set ({len(train_df)} rows)')
print(f'  ⚠️  Test set will be transformed using train statistics — NO leakage')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — MinMaxScaler (scales all features to range [0, 1])
#
# Formula : x' = (x - x_min) / (x_max - x_min)
#
# Where:
#   x'    = normalised value (output, between 0 and 1)
#   x     = original feature value
#   x_min = minimum value of that feature in the TRAINING set
#   x_max = maximum value of that feature in the TRAINING set
#
# Good for: features with known bounded ranges (like RSI 0-100, BB_Position 0-1)
# Sensitive to outliers — a single extreme value compresses everything else
#
# CRITICAL: fit() called on TRAIN only, transform() called on BOTH train and test
# ─────────────────────────────────────────────────────────────────────────

# Initialise scaler
minmax_scaler = MinMaxScaler(feature_range=(0, 1))

# FIT on training data only — learns x_min and x_max from train
train_minmax = minmax_scaler.fit_transform(train_df[FEATURE_COLS])

# TRANSFORM test data using training statistics — no refitting
test_minmax  = minmax_scaler.transform(test_df[FEATURE_COLS])

# Build scaled DataFrames
train_minmax_df = pd.DataFrame(train_minmax, columns=FEATURE_COLS)
train_minmax_df.insert(0, 'Date', train_df['Date'].values)
train_minmax_df.insert(1, 'Close', train_df['Close'].values)

test_minmax_df  = pd.DataFrame(test_minmax, columns=FEATURE_COLS)
test_minmax_df.insert(0, 'Date', test_df['Date'].values)
test_minmax_df.insert(1, 'Close', test_df['Close'].values)

# Combine train and test
all_minmax_df = pd.concat([train_minmax_df, test_minmax_df], ignore_index=True)

# Save scaler
joblib.dump(minmax_scaler, 'minmax_scaler.pkl')

print('=' * 60)
print('  MINMAX SCALER — FIT ON TRAIN, TRANSFORM BOTH')
print('=' * 60)
print(f'  Formula  : x\' = (x - x_min) / (x_max - x_min)')
print(f'  Range    : [0, 1]')
print(f'  Fit on   : {len(train_df)} training rows only')
print(f'  Scaler saved : minmax_scaler.pkl')
print()
print('  Min and Max learned from training data:')
print(f'  {"Feature":<20} {"Train Min":>12} {"Train Max":>12}')
print('  ' + '─' * 46)
for i, col in enumerate(FEATURE_COLS):
    fmin = minmax_scaler.data_min_[i]
    fmax = minmax_scaler.data_max_[i]
    print(f'  {col:<20} {fmin:>12.4f} {fmax:>12.4f}')

print()
print('  Sample scaled train values (last 3 rows):')
print(train_minmax_df[['Date'] + FEATURE_COLS[:4]].tail(3).round(4).to_string(index=False))
print()
print('  Verification — all values between 0 and 1:')
train_arr = train_minmax_df[FEATURE_COLS].values
test_arr  = test_minmax_df[FEATURE_COLS].values
print(f'    Train min : {train_arr.min():.6f}  Train max : {train_arr.max():.6f}')
print(f'    Test  min : {test_arr.min():.6f}  Test  max : {test_arr.max():.6f}')
if test_arr.min() < -0.01 or test_arr.max() > 1.01:
    print('    ⚠️  Test values slightly outside [0,1] — due to test data having')
    print('       values outside the training range (expected for out-of-sample data)')
else:
    print('    ✅ All values within [0, 1] range')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — StandardScaler (zero mean, unit standard deviation)
#
# Formula : z = (x - μ) / σ
#
# Where:
#   z  = standardised value (output)
#   x  = original feature value
#   μ  = mean of that feature in the TRAINING set
#   σ  = standard deviation of that feature in the TRAINING set
#
# Good for: normally distributed features, robust to outliers
# Output has no fixed range — can be negative or > 1
# Preferred for neural networks where activation functions expect zero-centred inputs
#
# CRITICAL: fit() called on TRAIN only, transform() called on BOTH train and test
# ─────────────────────────────────────────────────────────────────────────

# Initialise scaler
standard_scaler = StandardScaler()

# FIT on training data only — learns mean and std from train
train_standard = standard_scaler.fit_transform(train_df[FEATURE_COLS])

# TRANSFORM test data using training statistics — no refitting
test_standard  = standard_scaler.transform(test_df[FEATURE_COLS])

# Build scaled DataFrames
train_standard_df = pd.DataFrame(train_standard, columns=FEATURE_COLS)
train_standard_df.insert(0, 'Date', train_df['Date'].values)
train_standard_df.insert(1, 'Close', train_df['Close'].values)

test_standard_df  = pd.DataFrame(test_standard, columns=FEATURE_COLS)
test_standard_df.insert(0, 'Date', test_df['Date'].values)
test_standard_df.insert(1, 'Close', test_df['Close'].values)

# Combine train and test
all_standard_df = pd.concat([train_standard_df, test_standard_df], ignore_index=True)

# Save scaler
joblib.dump(standard_scaler, 'standard_scaler.pkl')

print('=' * 60)
print('  STANDARD SCALER — FIT ON TRAIN, TRANSFORM BOTH')
print('=' * 60)
print(f'  Formula  : z = (x - μ) / σ')
print(f'  Output   : mean ≈ 0, std ≈ 1 (no fixed range)')
print(f'  Fit on   : {len(train_df)} training rows only')
print(f'  Scaler saved : standard_scaler.pkl')
print()
print('  Mean and Std learned from training data:')
print(f'  {"Feature":<20} {"Train Mean":>12} {"Train Std":>12}')
print('  ' + '─' * 46)
for i, col in enumerate(FEATURE_COLS):
    fmean = standard_scaler.mean_[i]
    fstd  = standard_scaler.scale_[i]
    print(f'  {col:<20} {fmean:>12.4f} {fstd:>12.4f}')

print()
print('  Sample scaled train values (last 3 rows):')
print(train_standard_df[['Date'] + FEATURE_COLS[:4]].tail(3).round(4).to_string(index=False))
print()
print('  Verification — train mean ≈ 0, train std ≈ 1:')
train_means = train_standard_df[FEATURE_COLS].mean()
train_stds  = train_standard_df[FEATURE_COLS].std()
print(f'    Train mean (avg across features): {train_means.mean():.6f}  (should be ≈ 0)')
print(f'    Train std  (avg across features): {train_stds.mean():.6f}  (should be ≈ 1)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Data Leakage Verification
#
# This cell verifies that:
#   1. Scaler was fit ONLY on training data
#   2. Test data was only transformed (not fit)
#   3. No future information leaked into training
# ─────────────────────────────────────────────────────────────────────────

print('=' * 60)
print('  DATA LEAKAGE VERIFICATION')
print('=' * 60)

print()
print('  1. Train / Test date boundaries:')
print(f'     Last train date  : {train_df["Date"].iloc[-1].date()}')
print(f'     First test date  : {test_df["Date"].iloc[0].date()}')
gap = (test_df['Date'].iloc[0] - train_df['Date'].iloc[-1]).days
print(f'     Gap              : {gap} calendar days  ✅ Test starts after train ends')

print()
print('  2. MinMaxScaler was fit on training data only:')
print(f'     Scaler data_min_ shape : {minmax_scaler.data_min_.shape} (one min per feature)')
print(f'     These values come ONLY from {len(train_df)} training rows  ✅')

print()
print('  3. StandardScaler was fit on training data only:')
print(f'     Scaler mean_ shape : {standard_scaler.mean_.shape} (one mean per feature)')
print(f'     These values come ONLY from {len(train_df)} training rows  ✅')

print()
print('  4. Row count check:')
print(f'     Original clean rows : {len(df_clean)}')
print(f'     Train rows          : {len(train_df)}')
print(f'     Test rows           : {len(test_df)}')
print(f'     Train + Test        : {len(train_df) + len(test_df)}')
match = len(train_df) + len(test_df) == len(df_clean)
print(f'     Matches original    : {"✅ YES" if match else "❌ NO"}')

print()
print('  5. Inverse transform check (MinMax):')
# Take last row of train, inverse transform and compare to original
sample_scaled   = train_minmax[-1:]
sample_original = minmax_scaler.inverse_transform(sample_scaled)
original_vals   = train_df[FEATURE_COLS].iloc[-1].values
max_diff = np.max(np.abs(sample_original[0] - original_vals))
print(f'     Max inverse transform error : {max_diff:.8f}')
print(f'     Inverse transform working   : {"✅ YES" if max_diff < 0.01 else "❌ NO"}')

print()
print('  ✅ All leakage checks passed — data is clean and ready for model training')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Full Preview of Output
# ─────────────────────────────────────────────────────────────────────────

print('=' * 75)
print('  STEP 4 OUTPUT SUMMARY')
print('=' * 75)

print(f'\n  Input rows (after NaN drop) : {len(df_clean)}')
print(f'  Train rows                  : {len(train_df)}')
print(f'  Test rows                   : {len(test_df)}')
print(f'  Features scaled             : {len(FEATURE_COLS)}')

print(f'\n  MinMax Scaled — last 5 rows (train):')
print(train_minmax_df[['Date'] + FEATURE_COLS].tail(5).round(4).to_string(index=False))

print(f'\n  MinMax Scaled — first 5 rows (test):')
print(test_minmax_df[['Date'] + FEATURE_COLS].head(5).round(4).to_string(index=False))

print(f'\n  Standard Scaled — last 5 rows (train):')
print(train_standard_df[['Date'] + FEATURE_COLS].tail(5).round(4).to_string(index=False))

print(f'\n  Standard Scaled — first 5 rows (test):')
print(test_standard_df[['Date'] + FEATURE_COLS].head(5).round(4).to_string(index=False))

print()
print('  Which scaler to use for your model?')
print('  ┌─────────────────────────────────────────────────────────┐')
print('  │ MinMaxScaler   → Use when feeding into LSTM or FNN      │')
print('  │                  Good for sigmoid/tanh activations       │')
print('  │                  All values between 0 and 1             │')
print('  ├─────────────────────────────────────────────────────────┤')
print('  │ StandardScaler → Use when feeding into Transformer      │')
print('  │                  Good for GELU/ReLU activations          │')
print('  │                  Zero mean, unit std — no fixed range   │')
print('  └─────────────────────────────────────────────────────────┘')
print()
print('  For your project (Linear Transformer + FNN):')
print('  → Recommended: StandardScaler')
print('  → Both files are saved so you can experiment with either')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Save All Output Files and Download
# ─────────────────────────────────────────────────────────────────────────
import os

# ── Format dates for saving ──────────────────────────────────────────────
def format_dates(df_in):
    d = df_in.copy()
    d['Date'] = pd.to_datetime(d['Date']).dt.strftime('%d-%b-%Y')
    return d

# ── Save MinMax scaled files ─────────────────────────────────────────────
mm_train_path = 'nifty50_step4_minmax_train.csv'
mm_test_path  = 'nifty50_step4_minmax_test.csv'
mm_all_path   = 'nifty50_step4_minmax_scaled.csv'

format_dates(train_minmax_df).to_csv(mm_train_path, index=False)
format_dates(test_minmax_df).to_csv(mm_test_path,   index=False)
format_dates(all_minmax_df).to_csv(mm_all_path,     index=False)

# ── Save Standard scaled files ───────────────────────────────────────────
st_train_path = 'nifty50_step4_standard_train.csv'
st_test_path  = 'nifty50_step4_standard_test.csv'
st_all_path   = 'nifty50_step4_standard_scaled.csv'

format_dates(train_standard_df).to_csv(st_train_path, index=False)
format_dates(test_standard_df).to_csv(st_test_path,   index=False)
format_dates(all_standard_df).to_csv(st_all_path,     index=False)

# ── Save scalers ─────────────────────────────────────────────────────────
joblib.dump(minmax_scaler,   'minmax_scaler.pkl')
joblib.dump(standard_scaler, 'standard_scaler.pkl')

# ── Print summary ─────────────────────────────────────────────────────────
output_files = [
    (mm_train_path,          f'{len(train_df)} rows — MinMax scaled training data'),
    (mm_test_path,           f'{len(test_df)} rows  — MinMax scaled test data'),
    (mm_all_path,            f'{len(all_minmax_df)} rows — MinMax scaled full dataset'),
    (st_train_path,          f'{len(train_df)} rows — Standard scaled training data'),
    (st_test_path,           f'{len(test_df)} rows  — Standard scaled test data'),
    (st_all_path,            f'{len(all_standard_df)} rows — Standard scaled full dataset'),
    ('minmax_scaler.pkl',    'MinMaxScaler — use to inverse transform predictions'),
    ('standard_scaler.pkl',  'StandardScaler — use to inverse transform predictions'),
]

print('=' * 65)
print('  FILES SAVED')
print('=' * 65)
for fname, desc in output_files:
    size = os.path.getsize(fname)
    print(f'  ✓ {fname:<45} {size:>8} bytes')
    print(f'      {desc}')
print('=' * 65)

# ── Download all files ────────────────────────────────────────────────────
print('\n⬇️  Downloading all files...')
for fname, _ in output_files:
    files.download(fname)
    print(f'  ✓ {fname}')

print()
print('✅ Step 4 Complete!')
print()
print('  Files to use in your model:')
print('  → nifty50_step4_standard_train.csv  — feed into Linear Transformer training')
print('  → nifty50_step4_standard_test.csv   — feed into model evaluation')
print('  → standard_scaler.pkl               — use to inverse transform predictions')
print()
print('  Next step: Step 5 — Fetch News Data from GDELT')